# Extract Text from PDFs and DOCX Files


In [7]:
import os
import json
from pathlib import Path
from pypdf import PdfReader
from docx import Document

DATA_DIR = Path("./data")
OUTPUT_FILE = Path("./extracted_texts.json")

In [8]:
def extract_pdf(file_path: Path):
    reader = PdfReader(str(file_path))
    pages = []
    full_text_parts = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append({"page_number": i + 1, "text": text})
        full_text_parts.append(text)

    return {
        "filename": file_path.name,
        "source_type": "pdf",
        "total_pages": len(reader.pages),
        "full_text": "\n\n".join(full_text_parts),
        "pages": pages,
    }


def extract_docx(file_path: Path):
    doc = Document(str(file_path))
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]

    return {
        "filename": file_path.name,
        "source_type": "docx",
        "total_pages": None,
        "full_text": "\n\n".join(paragraphs),
        "pages": [],
    }


EXTRACTORS = {
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".doc": extract_docx,
}

In [9]:
files = sorted(DATA_DIR.iterdir()) if DATA_DIR.exists() else []
supported_files = [f for f in files if f.suffix.lower() in EXTRACTORS]

print(f"Found {len(supported_files)} supported files in {DATA_DIR}:")
for f in supported_files:
    print(f"  - {f.name} ({f.suffix})")

if not supported_files:
    print("\n⚠ No PDF/DOCX files found. Place your files in the ./data/ folder and re-run.")

Found 2 supported files in data:
  - 2507.19457v2.pdf (.pdf)
  - 2601.11868v1.pdf (.pdf)


In [5]:
extracted_documents = []

for file_path in supported_files:
    ext = file_path.suffix.lower()
    extractor = EXTRACTORS[ext]

    try:
        doc = extractor(file_path)
        char_count = len(doc["full_text"])
        extracted_documents.append(doc)
        print(f"OK  {file_path.name} — {char_count:,} chars extracted")
    except Exception as e:
        print(f"ERR {file_path.name} — {e}")

print(f"\nSuccessfully extracted {len(extracted_documents)}/{len(supported_files)} files")

OK  2507.19457v2.pdf — 274,406 chars extracted
OK  2601.11868v1.pdf — 202,157 chars extracted

Successfully extracted 2/2 files


In [6]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(extracted_documents, f, ensure_ascii=False, indent=2)

print(f"Saved {len(extracted_documents)} documents to {OUTPUT_FILE}")

for doc in extracted_documents:
    preview = doc["full_text"][:200].replace("\n", " ")
    print(f"\n--- {doc['filename']} ({doc['source_type']}) ---")
    print(f"Preview: {preview}...")

Saved 2 documents to extracted_texts.json

--- 2507.19457v2.pdf (pdf) ---
Preview: Accepted at ICLR 2026 (Oral). GEPA: REFLECTIVEPROMPTEVOLUTIONCANOUTPER- FORMREINFORCEMENTLEARNING Lakshya A Agrawal1, Shangyin Tan1, Dilara Soylu2, Noah Ziems4, Rishi Khare1,Krista Opsahl-Ong 5,Arnav ...

--- 2601.11868v1.pdf (pdf) ---
Preview: TERMINAL-BENCH: BENCHMARKINGAGENTS ONHARD, REALISTIC TASKS INCOMMANDLINEINTERFACES Mike A. Merrill1,*,Alexander G. Shaw 2,*,Nicholas Carlini 3,Boxuan Li 4,Harsh Raj 5,Ivan Bercovich6,Lin Shi 7,Jeong Y...
